# Notebook 2 — Facial fungal LoRA training | SD2.1 | Kaggle T4

**Train two separate LoRAs by running this notebook twice**:
1. `SUBTYPE='tinea_faciei'` and `IMAGE_ROOT` pointing **only to clinically verified tinea-faciei images**.
2. `SUBTYPE='pityriasis_versicolor'` and `IMAGE_ROOT` pointing **only to verified facial pityriasis-versicolor images**.

Do **not** train one LoRA on every folder of an unlabeled mixed fungal dataset and pretend it represents a specific disease. Real case provenance and subtype labels matter. **Facial candidiasis** is excluded until you audit enough correctly labelled training examples.

The model is **SD2.1 base + your own LoRA**. There is **no verified public fungal face LoRA** included. The `hcarrion/tinea_pedis` checkpoint must not be substituted for facial tinea.

Training uses the official Diffusers v0.35.1 text-to-image LoRA example with **single-chunk (≤75 content token) captions**. The 150-token Compel strategy applies only to the separate inference notebook.

## 1 | Install pinned packages

Start a Kaggle GPU T4 notebook with Internet ON. The install cell intentionally leaves Kaggle's GPU-enabled PyTorch alone. Restart the kernel if previous modules were loaded before installation.

In [ ]:
import sys, subprocess
PACKAGES = [
    'diffusers==0.35.1', 'transformers==4.56.2',
    'accelerate==1.10.1', 'peft==0.17.1',
    'datasets==3.6.0', 'huggingface_hub==0.34.4',
    'safetensors==0.6.2', 'ftfy==6.3.1',
    'imagehash==4.3.2', 'bitsandbytes==0.47.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES])
print('Pinned dependencies installed. Restart this session if imports had already been loaded.')

In [ ]:
import json, csv, hashlib, shutil, urllib.request, sys, subprocess, random
from pathlib import Path
from collections import Counter
import numpy as np
import imagehash
from PIL import Image, ImageOps
from sklearn.model_selection import GroupShuffleSplit
import torch, diffusers, transformers, accelerate, datasets, peft
from IPython.display import display
from huggingface_hub import model_info
assert torch.cuda.is_available(), 'Enable a Kaggle GPU (T4 recommended) in notebook settings.'
print('PyTorch', torch.__version__, '| CUDA', torch.cuda.get_device_name(0))
print('Diffusers', diffusers.__version__, '| Datasets', datasets.__version__)

## 2. Approved Kaggle real-image dataset v2
The sole raw real-image source is `/kaggle/input/datasets/shanmukhadattaboda/facial-skin-conditions-real-image-collection`. Original images in that source are **not directly passed into LoRA training**. Instead, attach the pre-reviewed, deduplicated frozen real train manifest, split images, and a passed real-label audit. Subtype-specific LoRA additionally requires a CSV with reviewer-approved image_id, subtype and reviewer_decision; never infer a precise subtype from the broad class folder. The notebook stages only train examples and streams trainer logs to Kaggle outputs. The optional run may fail if checkpoint/API dependencies or source permissions are unavailable.


**Updated split protocol:** this notebook explicitly reports the selected class's train/validation/test counts from the already reviewed GLOBAL image-group split, saves `class_frozen_split_counts.csv` and the private assignment CSV, and trains LoRA **only** on that class's training images. It will not create an independent raw-folder split that could leak repeated patients across classes. Complete console output is in `notebook_console.log` and the actual trainer log.


In [ ]:
SUBTYPE = 'tinea_faciei'    # one of: 'tinea_faciei', 'pityriasis_versicolor'
IMAGE_ROOT = Path('/kaggle/input/datasets/shanmukhadattaboda/facial-skin-conditions-real-image-collection/Face_Skin_dataset/Fungal_infection')  # REQUIRED EDIT: only this subtype
SPLIT_CSV = ''              # optional external CSV: filename,split
CAPTIONS_CSV = ''           # optional external CSV: filename,caption
AUTO_SPLIT = False          # fallback only; global patient grouping is preferable
BASE_MODEL = 'sd2-community/stable-diffusion-2-1-base'
OUT_DIR = Path('/kaggle/working/fungal_lora_output') / SUBTYPE
DATA_PREP = Path('/kaggle/working/fungal_lora_training_data') / SUBTYPE
STEPS = 20                 # first run; later inspect 300–1000 as experiments
RANK = 16
RESOLUTION = 512
BATCH_SIZE = 1
GRAD_ACCUM = 4
LR = 1e-4
SEED = 62031
CHECKPOINT_STEPS = 250
USE_8BIT_ADAM = True

if SUBTYPE not in {'tinea_faciei','pityriasis_versicolor'}:
    raise ValueError('Choose one supported subtype')
if not IMAGE_ROOT.is_dir():
    raise FileNotFoundError(f'Edit IMAGE_ROOT to the actual folder in Kaggle: {IMAGE_ROOT}')
assert STEPS >= 1 and RESOLUTION % 8 == 0 and RANK >= 1
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_PREP.mkdir(parents=True, exist_ok=True)
print('Subtype:', SUBTYPE, '| image folder:', IMAGE_ROOT)

In [ ]:
# Only Kaggle v2 real images reviewed and assigned to the FROZEN real TRAIN split.
from pathlib import Path
import sys,subprocess
DERMSYNREAL_REPO=Path('/kaggle/working/DermSynReal')
if not (DERMSYNREAL_REPO/'src/dermsynreal/kaggle_runs.py').is_file():
    subprocess.check_call(['git','clone','--depth','1','https://github.com/shanmukhdatta/DermSynReal.git',str(DERMSYNREAL_REPO)])
sys.path.insert(0,str(DERMSYNREAL_REPO/'src'))
from dermsynreal.kaggle_runs import prepare_lora_train_only,append_run_log
RAW_REAL_DATASET_V2='/kaggle/input/datasets/shanmukhadattaboda/facial-skin-conditions-real-image-collection'
REAL_TRAIN_MANIFEST='/kaggle/input/EDIT_APPROVED_REAL_SPLIT/real_all_manifest.csv'
REAL_TRAIN_IMAGE_ROOT='/kaggle/input/EDIT_APPROVED_REAL_IMAGES/facial_real_split_v2'
PASSED_REAL_LABEL_AUDIT='/kaggle/input/EDIT_REAL_LABEL_AUDIT/label_audit_summary.json'
REVIEWED_SUBTYPE_CSV='/kaggle/input/EDIT_REVIEWED_SUBTYPES/subtype_review.csv' # image_id,subtype,reviewer_decision
IMAGE_ROOT,real_train_evidence=prepare_lora_train_only(
    real_split_manifest=REAL_TRAIN_MANIFEST,real_image_root=REAL_TRAIN_IMAGE_ROOT,
    cls='Fungal_Infection',stage_dir='/kaggle/working/approved_Fungal_Infection_lora_input',
    origin_root=RAW_REAL_DATASET_V2,label_audit_json=PASSED_REAL_LABEL_AUDIT,subtype_csv=REVIEWED_SUBTYPE_CSV,subtype=SUBTYPE)
AUTO_SPLIT=False
SPLIT_CSV=''
# The global group-safe split is reused; show this class's training/validation/test counts.
print('Class split (before subtype selection):',real_train_evidence['class_frozen_split_counts'])
from dermsynreal.kaggle_runs import start_notebook_logging
PRIVATE_LORA_CONSOLE=start_notebook_logging(OUT_DIR,kind='lora_Fungal_Infection',
    config={'class_split_counts':real_train_evidence['class_frozen_split_counts'],
            'train_only_image_count':real_train_evidence['count'],
            'split_sha256':real_train_evidence['manifest_sha256']},repo_root=DERMSYNREAL_REPO)
print('Research console log:',PRIVATE_LORA_CONSOLE)
print('Approved train-only source:',IMAGE_ROOT,'images:',real_train_evidence['count'])
def run_logged(command,path):
    import subprocess
    from pathlib import Path
    log=Path(path);log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a',encoding='utf-8',errors='replace') as f:
        proc=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace',bufsize=1)
        for line in proc.stdout:
            print(line,end='');f.write(line);f.flush()
        code=proc.wait()
    if code:raise subprocess.CalledProcessError(code,command)


## 3 | Decode all images and calculate near-duplicate screening signals

The CPU preprocessing checks readability, canonical RGB, exact pixel hashes and flip-aware pHash before group splitting. It screens for near-duplicates but cannot prove patient identity.

In [ ]:
EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
all_images = sorted(p for p in IMAGE_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in EXTS)
if not all_images:
    raise ValueError(f'No JPG/PNG/WebP images inside {IMAGE_ROOT}')

records = []
for fp in all_images:
    try:
        with Image.open(fp) as img:
            norm = ImageOps.exif_transpose(img).convert('RGB')
            norm.load()
            h = imagehash.phash(norm)
            mirror_h = imagehash.phash(ImageOps.mirror(norm))
            rgb_hash = hashlib.sha256(norm.tobytes()).hexdigest()
        records.append({'path': fp, 'rel': fp.relative_to(IMAGE_ROOT).as_posix(),
                        'phash': h, 'mirror_phash': mirror_h, 'pixel_sha256': rgb_hash})
    except (OSError, ValueError) as exc:
        print('Skipping unreadable image:', fp, exc)
assert records, 'All images failed to decode'
print('Usable images:', len(records))

In [ ]:
# Complete-link exact and flip-aware pHash candidate grouping via union-find.
# pHash <=5 is a provisional near-duplicate trigger, NOT a medical or patient identity proof.
parent = list(range(len(records)))
def root(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

def union(i, j):
    ri, rj = root(i), root(j)
    if ri != rj:
        parent[rj] = ri

for i, a in enumerate(records):
    for j in range(i):
        b = records[j]
        is_exact = a['pixel_sha256'] == b['pixel_sha256']
        is_near = min(a['phash'] - b['phash'],
                      a['phash'] - b['mirror_phash'],
                      a['mirror_phash'] - b['phash'],
                      a['mirror_phash'] - b['mirror_phash']) <= 5
        if is_exact or is_near:
            union(i, j)
for i, r in enumerate(records):
    r['group'] = root(i)
print('Provisional groups:', len({r['group'] for r in records}))

## 4 | Protect validation and test partitions

If an explicit `train/` folder or split CSV is supplied, no image sharing a detected group with a held-out image enters LoRA training. In automatic mode, groups are partitioned before preparing training images.

In [ ]:
def csv_lookup(csv_file, field):
    if not csv_file:
        return {}
    out = {}
    with open(csv_file, newline='', encoding='utf-8-sig') as fh:
        reader = csv.DictReader(fh)
        if not {'filename', field}.issubset(reader.fieldnames or []):
            raise ValueError(f'{csv_file} must include filename,{field} columns')
        for row in reader:
            key = row['filename'].strip().replace('\\', '/')
            if key in out:
                raise ValueError(f'Duplicate filename in CSV: {key}')
            out[key] = row[field].strip()
    return out

split_lookup = csv_lookup(SPLIT_CSV, 'split')
caption_lookup = csv_lookup(CAPTIONS_CSV, 'caption')
basename_counts = Counter(Path(r['rel']).name for r in records)

def lookup(data, rec):
    if rec['rel'] in data:
        return data[rec['rel']]
    base = Path(rec['rel']).name
    if base in data:
        if basename_counts[base] != 1:
            raise ValueError(f'Ambiguous basename in metadata: {base}; use relative paths')
        return data[base]
    return None

train_path = IMAGE_ROOT / 'train'
if train_path.is_dir():
    for rec in records:
        rec['split'] = 'train' if train_path in rec['path'].parents else 'held_out'
    split_source = 'train/ directory'
elif SPLIT_CSV:
    for rec in records:
        value = lookup(split_lookup, rec)
        rec['split'] = (value or 'unassigned').lower()
    split_source = 'SPLIT_CSV'
elif AUTO_SPLIT:
    if len({r['group'] for r in records}) < 5:
        raise ValueError('Too few independent groups for provisional train/val/test split')
    indices = np.arange(len(records))
    groups = np.array([r['group'] for r in records])
    gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    train_ix, rest_ix = next(gss.split(indices, groups=groups))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED + 1)
    sub_val_ix, sub_test_ix = next(gss2.split(rest_ix, groups=groups[rest_ix]))
    val_ix, test_ix = rest_ix[sub_val_ix], rest_ix[sub_test_ix]
    train_set, val_set = set(train_ix), set(val_ix)
    for i, rec in enumerate(records):
        rec['split'] = ('train' if i in train_set else
                        'val' if i in val_set else 'test')
    split_source = 'AUTO_SPLIT provisional groups'
else:
    raise ValueError('Provide train/ or SPLIT_CSV, or enable AUTO_SPLIT')

# Never train on any group whose members were held out in a user-provided split.
held_groups = {r['group'] for r in records if r['split'] != 'train'}
train_records = [r for r in records if r['split'] == 'train' and r['group'] not in held_groups]
if not train_records:
    raise ValueError('No safe training images selected; review group overlaps and split CSV')

manifest = OUT_DIR / 'split_manifest.csv'
with manifest.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=['filename', 'group', 'split'])
    writer.writeheader()
    for r in records:
        writer.writerow({'filename': r['rel'], 'group': r['group'], 'split': r['split']})
print('Split policy:', split_source, '| train images:', len(train_records), '| manifest:', manifest)

## 5 | Create image/caption records and avoid face cropping

All images are normalized to RGB and padded rather than cropped; PNGs with transparency are composited against a neutral background to avoid black frames. Default captions state only the supplied **verified subtype**, not invented morphology. Add detailed captions for visible characteristics when you have reliable morphology annotations.

In [ ]:
# Rebuild generated training data to avoid accidentally mixing images from earlier runs.
# This only deletes the working cache, NEVER the source Kaggle input dataset.
if DATA_PREP.exists():
    shutil.rmtree(DATA_PREP)
images_dir = DATA_PREP / 'images'
images_dir.mkdir(parents=True, exist_ok=True)
meta = DATA_PREP / 'metadata.csv'
DEFAULT_CAPTIONS = {
    'tinea_faciei': ('Clinical facial photograph showing diagnosed tinea faciei, natural '
                     'skin texture and clinically plausible facial skin involvement'),
    'pityriasis_versicolor': ('Clinical facial photograph showing diagnosed pityriasis '
                            'versicolor, natural facial skin texture and pigment variation'),
}
with meta.open('w',newline='',encoding='utf-8') as fh:
    writer = csv.DictWriter(fh,fieldnames=['file_name','text'])
    writer.writeheader()
    for i, r in enumerate(train_records):
        name = f'{i:06d}.png'
        dst = images_dir / name
        with Image.open(r['path']) as raw:
            img = ImageOps.exif_transpose(raw)
            if img.mode in ('RGBA','LA') or 'transparency' in img.info:
                rgba = img.convert('RGBA')
                background = Image.new('RGBA',rgba.size,(215,215,215,255))
                background.alpha_composite(rgba)
                img = background.convert('RGB')
            else:
                img = img.convert('RGB')
            padded = ImageOps.pad(img,(RESOLUTION,RESOLUTION),
                                  method=Image.Resampling.LANCZOS,
                                  color=(215,215,215),centering=(0.5,0.5))
            padded.save(dst)
        caption = lookup(caption_lookup,r) or DEFAULT_CAPTIONS[SUBTYPE]
        writer.writerow({'file_name':f'images/{name}','text':caption})
with meta.open(newline='',encoding='utf-8') as fh:
    entries = list(csv.DictReader(fh))
assert len(entries) == len(train_records)
assert all((DATA_PREP / row['file_name']).is_file() and row['text'].strip() for row in entries)
print('Prepared',len(entries),'images; metadata:',meta)

## 6 | Validate the Hugging Face ImageFolder dataset and caption lengths

The real `datasets` reader verifies correct path references. Training captions are checked using the actual SD2.1 tokenizer and must fit 75 content tokens; inference uses separate Compel chunking.

In [ ]:
from datasets import load_dataset
sample = load_dataset('imagefolder', data_dir=str(DATA_PREP), split='train')
assert {'image', 'text'}.issubset(sample.column_names), sample.column_names
assert len(sample) == len(train_records), (len(sample), len(train_records))
print('ImageFolder successfully read:', len(sample), 'rows; columns:', sample.column_names)
display(sample[0]['image'].resize((256, 256)))
print('First caption:', sample[0]['text'])
# Validate caption length using the SD2.1 training tokenizer (single CLIP chunk).
from transformers import CLIPTokenizer
train_tokenizer = CLIPTokenizer.from_pretrained(BASE_MODEL,subfolder='tokenizer')
for i,caption in enumerate(sample['text']):
    n=len(train_tokenizer(caption,truncation=False,add_special_tokens=False)['input_ids'])
    if n>75:
        raise ValueError(f'Training caption {i} contains {n} content tokens (maximum 75); shorten it.')
print('All training captions fit SD2.1 training context (75 content tokens).')


## 7 | Download exact Diffusers v0.35.1 trainer

In [ ]:
SCRIPT_DIR = Path('/kaggle/working/diffusers_examples')
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)
SCRIPT = SCRIPT_DIR / 'train_text_to_image_lora.py'
SCRIPT_URL = ('https://raw.githubusercontent.com/huggingface/diffusers/v0.35.1/'
              'examples/text_to_image/train_text_to_image_lora.py')
if not SCRIPT.exists() or SCRIPT.stat().st_size < 10000:
    urllib.request.urlretrieve(SCRIPT_URL, SCRIPT)
script_text = SCRIPT.read_text(encoding='utf-8')
assert 'check_min_version("0.35.0")' in script_text
for arg in ('--train_data_dir', '--mixed_precision', '--use_8bit_adam', '--rank', '--gradient_checkpointing'):
    assert arg in script_text, f'Official script missing expected option {arg}'
print('Trainer script obtained:', SCRIPT, '| bytes:', SCRIPT.stat().st_size)

## 8 | Train the selected LoRA (20-step GPU smoke test first)

FP16 mixed precision is specified for both Accelerate and the official script, with 8-bit Adam and gradient checkpointing. The resulting LoRA is trained on your real **training partition only**. There is no pretrained facial-fungal adapter in the training step.

In [ ]:
base_revision = model_info(BASE_MODEL).sha
provenance = {
    'base_model': BASE_MODEL,
    'base_revision': base_revision,
    'adapter_type': 'custom_unet_lora',
    'subtype': SUBTYPE,
    'source_dataset': str(IMAGE_ROOT),
    'training_image_count': len(train_records),
    'resolution': RESOLUTION, 'rank': RANK, 'steps_requested': STEPS,
    'seed': SEED,
    'textual_inversion_used_during_training': False,
    'training_completed': False,
}
(OUT_DIR/'training_provenance.json').write_text(json.dumps(provenance,indent=2),encoding='utf-8')
command = [
    sys.executable, '-m', 'accelerate.commands.launch',
    '--mixed_precision', 'fp16', '--num_processes','1',
    str(SCRIPT),
    '--pretrained_model_name_or_path', BASE_MODEL,
    '--revision', base_revision,
    '--train_data_dir', str(DATA_PREP),
    '--image_column','image', '--caption_column','text',
    '--resolution',str(RESOLUTION), '--center_crop',
    '--train_batch_size',str(BATCH_SIZE),
    '--gradient_accumulation_steps',str(GRAD_ACCUM),
    '--max_train_steps',str(STEPS), '--learning_rate',str(LR),
    '--lr_scheduler','constant','--lr_warmup_steps','0',
    '--output_dir',str(OUT_DIR),
    '--checkpointing_steps',str(CHECKPOINT_STEPS),
    '--rank',str(RANK),'--seed',str(SEED),
    '--mixed_precision','fp16','--gradient_checkpointing',
    '--report_to','none',
]
if USE_8BIT_ADAM:
    command.append('--use_8bit_adam')
print('Launching trainer:', ' '.join(command))
run_logged(command,OUT_DIR/'trainer_console.log')

## 9 | Verify and export subtype LoRA checkpoint

For final generation, attach the ZIP to Kaggle and enter the `.safetensors` file path under the corresponding mode in Notebook 1. Retain the `training_provenance.json` file alongside the weights.

In [ ]:
weights = OUT_DIR/'pytorch_lora_weights.safetensors'
if not weights.is_file() or weights.stat().st_size == 0:
    raise RuntimeError(f'Training did not produce LoRA weights at {weights}')
# Mark provenance complete only after confirming the weight artifact exists.
provenance['training_completed'] = True
provenance['lora_sha256'] = hashlib.sha256(weights.read_bytes()).hexdigest()
(OUT_DIR/'training_provenance.json').write_text(json.dumps(provenance,indent=2),encoding='utf-8')
import zipfile
zip_path = Path('/kaggle/working') / f'{SUBTYPE}_sd21_lora_weights.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_STORED) as archive:
    archive.write(weights,weights.name)
    archive.write(OUT_DIR/'training_provenance.json','training_provenance.json')
print('Weights:',weights,'| export:',zip_path)

In [ ]:
# Retain the **three class split counts** beside the LoRA weight/checkpoint.
import shutil
from pathlib import Path
for _name in ('class_frozen_split_counts.csv','class_frozen_split_assignments.csv','lora_train_source_summary.json'):
    shutil.copy2(Path(IMAGE_ROOT)/_name,Path(OUT_DIR)/_name)
# Save LoRA checkpoint evidence without claiming it was medically validated.
append_run_log(OUT_DIR,kind='lora_fine_tuning',config={'class':'Fungal_Infection',
    'base_model':BASE_MODEL,'training_steps':STEPS,'seed':SEED,
    'approved_real_train_count':real_train_evidence['count'],
    'approved_manifest_sha256':real_train_evidence['manifest_sha256']},
    artifacts=[OUT_DIR/'trainer_console.log',
        OUT_DIR/'class_frozen_split_counts.csv',OUT_DIR/'pytorch_lora_weights.safetensors'],repo_root=DERMSYNREAL_REPO)
print('LoRA checkpoint and logs:',OUT_DIR)

## References and cautions

- [Official Diffusers v0.35.1 training script](https://raw.githubusercontent.com/huggingface/diffusers/v0.35.1/examples/text_to_image/train_text_to_image_lora.py)
- [cgDDI model collection](https://github.com/hectorcarrion/ControllableGenDDI): `tinea_pedis` is intentionally excluded from this facial-fungal generator.
- Training on a tiny, heterogeneous dataset can overfit and memorize identifiable faces. Review closest image matches before using synthetic data.
- Do not introduce candidiasis into the dataset without independently confirmed case labels and coherent morphology.